<style>
  @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@400;600;700&display=swap');
</style>

<div style="font-family:'Montserrat', ui-sans-serif, system-ui, -apple-system, 'Segoe UI', Roboto, Helvetica, Arial; padding:10px 0 6px 0; border-bottom:1px solid #ef4444;">
  <div style="display:flex; align-items:flex-start; gap:12px;">
    <img src="../assets/aiphet-logo.png" alt="Aiphet logo" style="width:44px;height:44px;border-radius:10px;object-fit:contain; margin-top:2px;" />
    <div style="line-height:1.15;">
      <div style="font-size:26px;font-weight:700;letter-spacing:0.2px;">Aiphet</div>
      <div style="font-size:14px;color:#4b5563;margin-top:2px;">Fine-tuning (LoRA/QLoRA) of Qwen-7B for high-quality PROMs, PREMs, and general medical forms</div>
      <div style="font-size:12px;color:#6b7280;margin-top:6px;">Author: Pablo Pimàs</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">Email: pablo@pimas.cat</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">Date: February 22, 2026</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">License: CC BY 4.0 (SPDX: CC-BY-4.0)</div>
      <div style="font-size:12px;color:#6b7280; font-weight:600; margin-top:2px;">#AIP-173</div>
    </div>
  </div>
</div>


# Fine-Tuning Qwen-7B for FHIR R4 QuestionnaireItem Generation

## Scope

This notebook documents an end-to-end, reproducible training workflow for adapting a Qwen-family 7B model to generate **FHIR R4 QuestionnaireItem-like JSON** from ChatML-formatted prompts in a clinical PROMs/PREMs context.

The workflow is designed for:

- Apple Silicon execution using MLX/MLX-LM
- parameter-efficient adaptation (LoRA/QLoRA-style setup)
- structured-output generation under FHIR-oriented constraints

## Main Contributions

1. Reproducible data preparation and validation pipeline for JSONL + ChatML records.
2. Structured fine-tuning protocol for FHIR Questionnaire item generation.
3. Quantitative evaluation of structural validity and schema-conformance behavior.
4. Qualitative error analysis of generated medical questionnaire items.
5. Explicit legal/licensing caveats for standardized instrument content.

## Research Context

Patient-reported outcomes (PROMs) and patient-reported experience measures (PREMs) often require strict structural interoperability.  
This notebook focuses on generating machine-usable questionnaire items compatible with **HL7 FHIR R4 Questionnaire semantics**.

## References (Core Concepts)

- HL7 FHIR R4 Questionnaire: https://hl7.org/fhir/R4/questionnaire.html
- LoRA (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- QLoRA (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314
- Qwen documentation: https://qwen.readthedocs.io/
- MLX: https://github.com/ml-explore/mlx
- MLX-LM: https://github.com/ml-explore/mlx-lm

## Compliance Note

This notebook is provided for research and educational use.  
If standardized questionnaire text is redistributed or used commercially, licensing and copyright obligations must be verified instrument-by-instrument.

## Reproducibility Protocol

To support repeatability, this notebook follows a deterministic workflow where possible.

### Execution Environment

- Platform target: Apple Silicon (macOS) with MLX/MLX-LM
- Python environment managed with pinned package versions from `requirements.txt`
- Notebook and scripts executed from repository root to preserve relative paths

### Determinism Settings

We will enforce and report:

- global random seed
- dataset split seed
- explicit train/validation partition method
- fixed configuration values for model and optimization

> Note: exact bitwise reproducibility is not always guaranteed across hardware/software backends, but seed and environment control substantially reduce variance.

### Experimental Tracking

For each training run, we will record:

1. model identifier
2. dataset file path and row counts
3. preprocessing/validation checks
4. hyperparameters (learning rate, LoRA rank, batch size, sequence length, steps)
5. runtime metadata (device, library versions)
6. evaluation outputs and qualitative examples

### Run Order

This notebook should be executed top-to-bottom in a single pass:

1. environment checks
2. dataset loading and validation
3. split generation
4. training
5. evaluation and error analysis

### References

- ACM Artifact Review and Badging: https://www.acm.org/publications/policies/artifact-review-and-badging-current
- The Turing Way (Reproducibility): https://the-turing-way.netlify.app/reproducible-research/reproducible-research
- MLX: https://github.com/ml-explore/mlx

In [3]:
# Reproducibility: environment and package snapshot

from __future__ import annotations

import importlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

# --- Fixed seeds for reproducibility ---
GLOBAL_SEED = 173
random.seed(GLOBAL_SEED)
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)

# --- Helper to get package versions safely ---
def safe_version(module_name: str) -> str:
    try:
        module = importlib.import_module(module_name)
        return getattr(module, "__version__", "unknown")
    except Exception:
        return "not-installed"

# --- Detect repository root heuristically ---
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return start.resolve()

repo_root = find_repo_root(Path.cwd())
dataset_path = repo_root / "data" / "aiprom-dataset-fhir-4-150.jsonl"

# --- System/runtime info ---
runtime_info = {
    "timestamp_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cwd": str(Path.cwd().resolve()),
    "repo_root": str(repo_root),
    "dataset_exists": dataset_path.exists(),
    "dataset_path": str(dataset_path),
    "seed": GLOBAL_SEED,
    "packages": {
        "mlx": safe_version("mlx"),
        "mlx_lm": safe_version("mlx_lm"),
        "datasets": safe_version("datasets"),
        "yaml": safe_version("yaml"),
        "numpy": safe_version("numpy"),
        "pandas": safe_version("pandas"),
        "wandb": safe_version("wandb"),
    },
}

# --- Optional: git metadata if available ---
if shutil.which("git"):
    try:
        commit = subprocess.check_output(
            ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
            text=True
        ).strip()
    except Exception:
        commit = "unavailable"
else:
    commit = "git-not-found"

runtime_info["git_commit"] = commit

print(json.dumps(runtime_info, indent=2))

{
  "timestamp_utc": "2026-02-22T14:22:33Z",
  "python_version": "3.14.3",
  "platform": "macOS-26.3-arm64-arm-64bit-Mach-O",
  "machine": "arm64",
  "processor": "arm",
  "cwd": "/Users/CAE9/aiprom-llm/lab",
  "repo_root": "/Users/CAE9/aiprom-llm",
  "dataset_exists": true,
  "dataset_path": "/Users/CAE9/aiprom-llm/data/aiprom-dataset-fhir-4-150.jsonl",
  "seed": 173,
  "packages": {
    "mlx": "unknown",
    "mlx_lm": "0.30.7",
    "datasets": "4.5.0",
    "yaml": "6.0.3",
    "numpy": "2.4.2",
    "pandas": "3.0.1",
    "wandb": "0.25.0"
  },
  "git_commit": "2933566cfeb5f782cc70122ab179cfb02ff189ff"
}


/var/folders/vp/yhv0twn94p76ss5nftb0d5580000gp/T/ipykernel_65550/3300354309.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",


## Dataset Statement and Governance

### Primary Dataset

This study uses the JSONL dataset:

- `data/aiprom-dataset-fhir-4-150.jsonl`

Each row contains a ChatML transcript in a single `text` field, with a structured assistant target representing a **FHIR R4 QuestionnaireItem-like JSON fragment**.

### Structural Coverage

The dataset includes all 13 Questionnaire item types used in this project:

- `group`, `display`, `boolean`, `decimal`, `integer`, `date`, `dateTime`, `time`, `string`, `text`, `url`, `choice`, `open-choice`

### Clinical Content Scope

Prompt/content coverage includes validated PROMs-related instruments and clinical scales (e.g., EORTC QLQ-C30, PHQ-9, GAD-7, EQ-5D-5L, PROMIS, WPAI, HADS), plus symptom-oriented constructs such as pain, fatigue, anxiety, depression, and quality of life.

### Data Format Contract

The training objective is conditioned generation:

- **Input**: system + user turns in ChatML
- **Output**: assistant JSON with FHIR-oriented fields (`linkId`, `text`, `type`, `required`, `code`, and `answerOption` when applicable)

### Licensing and Usage Caveat

This notebook is for research/educational purposes.  
Some questionnaire content may correspond to third-party instruments with distinct copyright or licensing terms.

Before redistribution or commercial use, verify rights instrument-by-instrument.

See repository documentation for details:

- `README.md` (licensing overview and practical implications)
- `docs/datasets.md` (dataset scope and legal notes)

### References

- FHIR R4 Questionnaire: https://hl7.org/fhir/R4/questionnaire.html
- FAIR principles for scientific data: https://www.go-fair.org/fair-principles/
- PROMIS (HealthMeasures): https://www.healthmeasures.net/explore-measurement-systems/promis